In [4]:
import pandas as pd
import sqlalchemy as sa

engine = sa.create_engine(
    "mssql+pyodbc://@.\\SQLEXPRESS/Projects"
    "?driver=ODBC+Driver+17+for+SQL+Server"
    "&trusted_connection=yes"
    "&TrustServerCertificate=yes"
)

In [2]:
#验证数据库是否联通
# pd.read_sql("SELECT 1 AS test_col", engine)

In [5]:
#验证表是否存在
pd.read_sql("""
SELECT TABLE_NAME
FROM INFORMATION_SCHEMA.TABLES
WHERE TABLE_TYPE='BASE TABLE'
""", engine)

C:\Users\Administrator\AppData\Local\Programs\Python\Python314\Lib\site-packages\pandas\io\sql.py:1648: SAWarning: Unrecognized server version info '17.0.1000.7'.  Some SQL Server features may not function properly.
  con = self.exit_stack.enter_context(con.connect())


,TABLE_NAME
0,df_orders


In [6]:
#查看表的前5行
pd.read_sql("SELECT TOP 5 * FROM df_orders", engine)

,order_id,order_date,ship_mode,segment,country,city,state,postal_code,region,category,sub_category,product_id,quantity,discount,sale_price,profit
0,3598,2022-08-17,Second Class,Consumer,United States,Philadelphia,Pennsylvania,19140,East,Technology,Accessories,TEC-AC-10002134,6,1.8,58.2,8.2
1,3599,2022-11-09,Second Class,Consumer,United States,Philadelphia,Pennsylvania,19140,East,Furniture,Furnishings,FUR-FU-10002456,3,0.6,19.4,-0.6
2,3600,2022-11-07,Second Class,Consumer,United States,Philadelphia,Pennsylvania,19140,East,Technology,Phones,TEC-PH-10002549,1,4.2,135.8,5.8
3,3601,2023-04-01,Standard Class,Consumer,United States,Denver,Colorado,80219,West,Technology,Accessories,TEC-AC-10001990,9,21.5,408.5,58.5
4,3602,2023-04-08,Standard Class,Consumer,United States,Philadelphia,Pennsylvania,19143,East,Office Supplies,Paper,OFF-PA-10002137,2,0.3,9.7,-0.3


In [7]:
#总行数
pd.read_sql("SELECT COUNT(*) AS tatal_rows FROM df_orders", engine)

,tatal_rows
0,9994


In [8]:
#统计去重产品数量
query = """
SELECT COUNT(DISTINCT product_id) AS unique_product_count
FROM df_orders
"""
pd.read_sql(query, engine)

,unique_product_count
0,1862


In [9]:
#有哪些地区，从多到少排列
query = """
SELECT DISTINCT region
FROM df_orders
ORDER BY region DESC
"""
pd.read_sql(query, engine)

,region
0,West
1,South
2,East
3,Central


In [10]:
#哪些年份
query = """
SELECT DISTINCT YEAR(order_date) AS order_year
FROM df_orders
ORDER BY order_year
"""
pd.read_sql(query, engine)

,order_year
0,2022
1,2023


In [ ]:
#总销售额前十产品，从高到低
#query = """
#SELECT TOP 10
#    product_id,
#    SUM(sale_price) AS total_sales
#FROM df_orders
#GROUP BY product_id
#ORDER BY total_sales DESC
#"""
#pd.read_sql(query, engine)

In [ ]:
#总利润前十的产品
# query = """
# SELECT TOP 10
#    product_id,
#   SUM(profit) AS total_profit
#FROM df_orders
#GROUP BY product_id
#ORDER BY total_profit DESC
#"""
#pd.read_sql(query, engine)

In [11]:
#计算对比总销售额和总利润前十的产品，记录各自排行并分析利润与销量的关系
#用窗口函数RANK() OVER ()
query = """
WITH ranked AS (
    SELECT 
        product_id,
        SUM(sale_price) AS total_sales,
        SUM(profit) AS total_profit,
        RANK() OVER (ORDER BY SUM(sale_price) DESC) AS sales_rank,
        RANK() OVER (ORDER BY SUM(profit) DESC) AS profit_rank

    FROM df_orders
    GROUP BY product_id
)

SELECT *,
    CASE 
        WHEN sales_rank <= 10 AND profit_rank <= 10 THEN 'High Sales & High Profit'
        WHEN sales_rank <= 10 THEN 'High Sales Only'
        WHEN profit_rank <= 10 THEN 'High Profit Only'
    END AS product_type

FROM ranked
WHERE sales_rank <= 10 OR profit_rank <= 10;
"""
pd.read_sql(query, engine)

,product_id,total_sales,total_profit,sales_rank,profit_rank,product_type
0,TEC-CO-10004722,59514.0,5644.0,1,1,High Sales & High Profit
1,TEC-MA-10002412,21734.4,3624.4,3,2,High Sales & High Profit
2,OFF-BI-10003527,26525.3,3435.3,2,3,High Sales & High Profit
3,TEC-CO-10001449,18151.2,2631.2,7,4,High Sales & High Profit
4,FUR-CH-10002024,21096.2,2246.2,4,5,High Sales & High Profit
5,OFF-BI-10001359,19090.2,2080.2,5,6,High Sales & High Profit
6,OFF-BI-10000545,18249.0,1959.0,6,7,High Sales & High Profit
7,OFF-BI-10001120,15505.7,1695.7,13,8,High Profit Only
8,OFF-BI-10004995,17354.8,1654.8,9,9,High Sales & High Profit
9,FUR-BO-10004834,15024.1,1614.1,14,10,High Profit Only


总体来讲，销售额靠前产品的利润都比较高，但是根据下图可以发现，销售额高的产品利润高的主要原因是销售量大，并不是利润率高的原因

In [12]:
#以上我们可以明显看出有些产品销量和利润不匹配的问题，以下开始查找问题
#查看产品总销量总利润和利润率的关系
#利润率  利润率=利润÷销售额，销售额=单价×数量
query = """
WITH product_metrics AS (
    SELECT 
        product_id,
        SUM(sale_price) AS total_sales,
        SUM(profit) AS total_profit,
        SUM(profit) * 1.0 / NULLIF(SUM(sale_price * quantity), 0) AS profit_margin
        
    FROM df_orders
    GROUP BY product_id
)

SELECT  TOP 30 *
FROM product_metrics
ORDER BY total_sales DESC;
"""
pd.read_sql(query, engine)

,product_id,total_sales,total_profit,profit_margin
0,TEC-CO-10004722,59514.0,5644.0,0.023031
1,OFF-BI-10003527,26525.3,3435.3,0.035385
2,TEC-MA-10002412,21734.4,3624.4,0.027793
3,FUR-CH-10002024,21096.2,2246.2,0.018704
4,OFF-BI-10001359,19090.2,2080.2,0.029153
5,OFF-BI-10000545,18249.0,1959.0,0.011961
6,TEC-CO-10001449,18151.2,2631.2,0.024502
7,TEC-MA-10001127,17906.4,1106.4,0.014247
8,OFF-BI-10004995,17354.8,1654.8,0.022135
9,OFF-SU-10000151,16325.8,1215.8,0.018838


结论:由于数据中缺乏成本和物流信息，无法直接判断利润低的具体原因，但从利润率分析来看，这一批产品的利润率整体很低
在1%和3%之间徘徊，明显低于常见商业场景（通常为 10%+）

In [13]:
# 每个地区前5高销售产品
#WITH开始的CTE公共临时表, WITH = 给复杂SQL分步骤写
#基于原表，  第一个CTE 临时表格每个地区每个产品的销售额算出来，生成临时结果  
#第二个CTE基于第一个CTE，   对每个地区内的产品按销售额进行排名，并生成排名列 rn
#最后基于第二个CTE筛选出每个地区排名前5的产品
query = """
WITH region_product_sales AS (
    SELECT
        region,
        product_id,
        SUM(sale_price) AS total_sales
    FROM df_orders
    GROUP BY region, product_id
),
ranked_sales AS (
    SELECT
        region,
        product_id,
        total_sales,
        ROW_NUMBER() OVER (
            PARTITION BY region
            ORDER BY total_sales DESC
        ) AS rn
    FROM region_product_sales
)
SELECT
    region,
    product_id,
    total_sales,
    rn
FROM ranked_sales
WHERE rn <= 5
ORDER BY region, rn
"""
pd.read_sql(query, engine)

,region,product_id,total_sales,rn
0,Central,TEC-CO-10004722,16975.0,1
1,Central,TEC-MA-10000822,13770.0,2
2,Central,OFF-BI-10001120,11056.5,3
3,Central,OFF-BI-10000545,10132.7,4
4,Central,OFF-BI-10004995,8416.1,5
5,East,TEC-CO-10004722,29099.0,1
6,East,TEC-MA-10001047,13767.0,2
7,East,FUR-BO-10004834,11274.1,3
8,East,OFF-BI-10001359,8463.6,4
9,East,TEC-CO-10001449,8316.0,5


每个地区销售额前五的产品不同，说明每个地区的消费者的喜好不同,因此针对不同地区应该有不同的营销策略

In [23]:
#每个地区每年销售额最高的类别
#窗口函数ROW_NUMBER()排名不重复，  PARTITION BY region 根据地区分割
#只要一个 → ROW_NUMBER 
#允许并列 → RANK / DENSE_RANK
query = """
WITH category_region_sales AS (
    SELECT
        region,
        category,
        SUM(sale_price) AS total_sales
    FROM df_orders
    GROUP BY region,category
),
ranked_category_sales AS (
    SELECT
        region,
        category,
        total_sales,
        ROW_NUMBER() OVER (   
            PARTITION BY region 
            ORDER BY total_sales DESC
        ) AS rn
    FROM category_region_sales
)
SELECT
    region,
    category,
    total_sales
FROM ranked_category_sales
WHERE rn = 1
ORDER BY region
"""
pd.read_sql(query, engine)

,region,category,total_sales
0,Central,Technology,164552.6
1,East,Technology,255581.8
2,South,Technology,143606.4
3,West,Furniture,243571.4


In [ ]:
每个地区销售额最高的类别不同而且销售额有较大的差距

In [16]:
# 2022 和 2023 每月销售额对比
query = """
WITH monthly_sales AS (
    SELECT
        YEAR(order_date) AS order_year,
        MONTH(order_date) AS order_month,
        SUM(sale_price) AS total_sales
    FROM df_orders
    GROUP BY YEAR(order_date), MONTH(order_date)
)
SELECT
    order_month,
    SUM(CASE WHEN order_year = 2022 THEN total_sales ELSE 0 END) AS sales_2022,
    SUM(CASE WHEN order_year = 2023 THEN total_sales ELSE 0 END) AS sales_2023
FROM monthly_sales
GROUP BY order_month
ORDER BY order_month
"""
pd.read_sql(query, engine)

,order_month,sales_2022,sales_2023
0,1,94712.5,88632.6
1,2,90091.0,128124.2
2,3,80106.0,82512.3
3,4,95451.6,111568.6
4,5,79448.3,86447.9
5,6,94170.5,68976.5
6,7,78652.2,90563.8
7,8,104808.0,87733.6
8,9,79142.2,76658.6
9,10,118912.7,121061.5


In [17]:
#每个类别category销售额最高的月份
query = """
WITH category_month_sales AS (
    SELECT
        category,
        FORMAT(order_date, 'yyyy-MM') AS order_year_month,
        SUM(sale_price) AS total_sales
    FROM df_orders
    GROUP BY category, FORMAT(order_date, 'yyyy-MM')
),
ranked_category_sales AS (
    SELECT
        category,
        order_year_month,
        total_sales,
        ROW_NUMBER() OVER (
            PARTITION BY category
            ORDER BY total_sales DESC
        ) AS rn
    FROM category_month_sales
)
SELECT
    category,
    order_year_month,
    total_sales
FROM ranked_category_sales
WHERE rn = 1
ORDER BY category
"""
pd.read_sql(query, engine)

,category,order_year_month,total_sales
0,Furniture,2022-10,42888.9
1,Office Supplies,2023-02,44118.5
2,Technology,2023-10,53000.1


In [18]:
#2023比2022销售额增长最多的子类别sub_category
query = """
WITH subcategory_year_sales AS (
    SELECT
        sub_category,
        YEAR(order_date) AS order_year,
        SUM(sale_price) AS total_sales
    FROM df_orders
    GROUP BY sub_category, YEAR(order_date)
),
subcategory_compare AS (
    SELECT
        sub_category,
        SUM(CASE WHEN order_year = 2022 THEN total_sales ELSE 0 END) AS sales_2022,
        SUM(CASE WHEN order_year = 2023 THEN total_sales ELSE 0 END) AS sales_2023
    FROM subcategory_year_sales
    GROUP BY sub_category
)
SELECT TOP 1
    sub_category,
    sales_2022,
    sales_2023,
    (sales_2023 - sales_2022) AS sales_growth
FROM subcategory_compare
ORDER BY sales_growth DESC
"""
pd.read_sql(query, engine)

,sub_category,sales_2022,sales_2023,sales_growth
0,Machines,73723.2,109178.5,35455.3


2023年增长最多的子类别是Machines